# EduAI: QLoRA reference path (CUDA / Colab)

This notebook is the CUDA counterpart of the MLX fine-tune in `configs/lora_llama32_3b.yaml`. **It has not been run as part of this repository's results**; the committed numbers all come from the MLX run on an M1 Pro. It is here so the training can be reproduced on a T4 or better.

- Base model: `unsloth/Llama-3.2-3B-Instruct` (ungated mirror, Llama 3.2 Community License).
- 4-bit NF4 quantization with fp16 compute, LoRA rank 8 on all linear layers of the last 16 blocks.
- TRL `SFTTrainer` on a conversational prompt/completion dataset, so the loss covers only the assistant completion (the MLX run uses `mask_prompt: true` for the same effect). This avoids `assistant_only_loss`, which needs `{% generation %}` tags that the Llama 3.2 chat template lacks.
- The SFT files come from `eduai data build`, which uses the same `prompts.py` as inference. Upload `data/sft/train.jsonl` and `valid.jsonl` (they are not in git; SciQ is CC BY-NC).

In [ ]:
!pip install -q transformers==5.17.0 peft==0.21.0 trl==1.13.0 bitsandbytes==0.50.2 datasets==5.0.1 accelerate==1.15.0

In [ ]:
import json
from datasets import Dataset

def to_prompt_completion(path):
    rows = []
    for line in open(path):
        msgs = json.loads(line)["messages"]
        rows.append({"prompt": msgs[:-1], "completion": [msgs[-1]]})
    return Dataset.from_list(rows)

train = to_prompt_completion("train.jsonl")
valid = to_prompt_completion("valid.jsonl")
len(train), len(valid)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

base = "unsloth/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map="auto")

n_layers = model.config.num_hidden_layers
lora = LoraConfig(r=8, lora_alpha=160, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                  layers_to_transform=list(range(n_layers - 16, n_layers)))
# lora_alpha = 20 * r matches mlx-lm's scale 20.0 (mlx applies scale directly; PEFT uses alpha / r).

In [ ]:
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(output_dir="llama32-3b-eduai-qlora", per_device_train_batch_size=4, gradient_accumulation_steps=1,
                max_steps=600, learning_rate=2e-5, lr_scheduler_type="constant", logging_steps=10,
                eval_strategy="steps", eval_steps=100, save_steps=100, max_length=1024, fp16=True,
                gradient_checkpointing=True, completion_only_loss=True, seed=7, report_to=[])
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train, eval_dataset=valid, processing_class=tok,
                     peft_config=lora)
trainer.train()
trainer.save_model("llama32-3b-eduai-qlora/final")

The saved PEFT adapter can be served with `eduai.llm.hf_backend.HFBackend(base, adapter=...)`. It is not interchangeable with the MLX adapter file format.